# HAM10000 - FedBN extension (secondary analysis)

Conditional extension from the proposal (Sec. IV-D, FL-006). **FedBN** averages every parameter like FedAvg **except BatchNorm** (weights, biases, running statistics), which stays on each client. The result is **one model per client** (shared weights + that client's BN), so - as in the FedBN paper - the natural evaluation is per client. It is reported as secondary analysis and does not change the main comparison in `03_federated_training.ipynb`.

Everything else is identical to the main FedAvg arm: EfficientNet-B0 (primary backbone by the pre-specified rule), same partition, augmentation, loss, optimizer, R = 20 rounds × E = 1, seeds 42/43/44. Model selection: best round by the **mean pooled-validation Macro-F1 of the 4 client models**; test predicted once.

**Kaggle setup:** GPU T4 x2, Internet on, inputs `kmader/skin-cancer-mnist-ham10000` + `HAM10000_fl_partition_v1`. **Optional but recommended:** also add `ham10000_fl_results.zip` (output of notebook 03) as a dataset; Kaggle unzips it, and this notebook then compares FedBN with FedAvg, Local-only and Centralized on the same test images. Expected time: ~25 min.

In [1]:
import glob
import json
import os
import queue
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

INPUT_ROOT = Path(os.getenv("HAM_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.getenv("HAM_WORK_DIR", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def find_one(name):
    hits = sorted(glob.glob(str(INPUT_ROOT / "**" / name), recursive=True))
    assert hits, f"{name} not found under {INPUT_ROOT}"
    return Path(hits[0])

PARTITION_PATH = find_one("HAM10000_fl_partition_v1.csv")
CACHE_PATH = WORK / "cache" / "ham10000_224_uint8.npy"
RUNS_DIR, LOG_DIR, RESULTS_DIR = WORK / "runs", WORK / "logs", WORK / "results_fedbn"
for d in (CACHE_PATH.parent, RUNS_DIR, LOG_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Results of notebook 03 (optional): the folder that contains fedavg_efficientnet_b0_s42/
prev = sorted(glob.glob(str(INPUT_ROOT / "**" / "fedavg_efficientnet_b0_s42" / "preds.npz"), recursive=True))
PREV_RUNS = Path(prev[0]).parent.parent if prev else None

BACKBONE = "efficientnet_b0"   # primary backbone chosen in notebook 03 (backbone_decision.json)
EPOCHS, LOCAL_EPOCHS, SEEDS = 20, 1, [42, 43, 44]
N_BOOTSTRAP = 1000

DEBUG = os.getenv("HAM_DEBUG") == "1"
EXTRA_ARGS = ["--debug-n", "8", "--no-pretrained", "--batch-size", "8"] if DEBUG else []
if DEBUG:
    EPOCHS, SEEDS, N_BOOTSTRAP = 2, [42, 43], 50

N_GPUS = torch.cuda.device_count()
SLOTS = max(N_GPUS, 1)
print(f"Partition: {PARTITION_PATH}")
print(f"Notebook-03 runs for comparison: {PREV_RUNS or 'not attached (FedBN-only tables)'}")
print(f"GPUs: {N_GPUS} | torch {torch.__version__} | DEBUG={DEBUG}")

Partition: /kaggle/input/datasets/kietvgpham/ham10000-fl-partition-v1/HAM10000_fl_partition_v1.csv
Notebook-03 runs for comparison: /kaggle/input/datasets/kietvgpham/ham10000-fl-results/runs
GPUs: 2 | torch 2.11.0+cu130 | DEBUG=False


## 1. Training script (same as notebook 03 + the `fedbn` arm)

In [2]:
%%writefile fl_train.py
#!/usr/bin/env python
"""Train one arm of the HAM10000 federated-learning study.

Arms (all share backbone, 224x224 input, augmentation, loss, optimizer and budget):
  centralized  pooled training split of all 4 clients
  local        one client only (--client)
  fedavg       FedAvg, sample-weighted averaging, all clients every round
  fedprox      FedAvg + proximal term (--mu)
  fedbn        FedAvg that keeps BatchNorm layers local (one model per client)

Budget rule (proposal, Sec. V-C): R rounds x E local epochs = centralized epochs,
so every arm makes the same number of passes over each training image.

Model selection uses the pooled validation set only. The test set is predicted
once, with the best checkpoint, at the end of the run.

Usage:
  python fl_train.py --build-cache --partition P --image-root R --cache C
  python fl_train.py --arm fedavg --partition P --cache C --out-dir runs
"""
import argparse
import hashlib
import json
import math
import os
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASSES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
CLIENTS = ["vidir_molemax", "vidir_modern", "rosendahl", "vienna_dias"]
FL_COLUMNS = ["image_id", "lesion_id", "dx", "split", "client"]
FL_MANIFEST_SHA256 = "0ab49e23b128c7d9f2d309bee762ade1d6ab69f04a31ca94096f29484c9a033d"
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# --------------------------------------------------------------------------- data
def fingerprint(df, cols):
    """Same definition as 02_federated_client_partition.ipynb."""
    csv = df[cols].sort_values("image_id").to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(csv.encode("utf-8")).hexdigest()


def load_partition(path, check=True):
    part = pd.read_csv(path)
    assert list(part.columns) == FL_COLUMNS, part.columns.tolist()
    if check:
        sha = fingerprint(part, FL_COLUMNS)
        if sha != FL_MANIFEST_SHA256:
            sys.exit(f"FL partition fingerprint mismatch: {sha}. Do not train on this file.")
    return part.sort_values("image_id").reset_index(drop=True)


def _load_one(path):
    from PIL import Image
    with Image.open(path) as im:
        im = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return np.asarray(im, dtype=np.uint8).transpose(2, 0, 1).copy()


def build_cache(partition_path, image_root, cache_path):
    """Resize every image once to 224x224 (proposal: 'resized to 224 x 224')."""
    from multiprocessing import Pool

    part = load_partition(partition_path)
    paths = {}
    for p in Path(image_root).rglob("ISIC_*.jpg"):
        paths.setdefault(p.stem, p)  # the Kaggle copy has duplicate folders; any copy is fine
    missing = [i for i in part["image_id"] if i not in paths]
    assert not missing, f"{len(missing)} images not found under {image_root}, e.g. {missing[:3]}"

    files = [str(paths[i]) for i in part["image_id"]]
    X = np.lib.format.open_memmap(cache_path, mode="w+", dtype=np.uint8,
                                  shape=(len(files), 3, IMG_SIZE, IMG_SIZE))
    t0 = time.time()
    with Pool(os.cpu_count()) as pool:
        for i, arr in enumerate(pool.imap(_load_one, files, chunksize=32)):
            X[i] = arr
            if (i + 1) % 2000 == 0:
                print(f"  cached {i + 1:,}/{len(files):,} ({time.time() - t0:.0f}s)", flush=True)
    X.flush()
    Path(str(cache_path) + ".ids.txt").write_text("\n".join(part["image_id"]) + "\n")
    print(f"Cache written: {cache_path} {X.shape} in {time.time() - t0:.0f}s")


def load_cache(cache_path, part):
    ids = Path(str(cache_path) + ".ids.txt").read_text().split()
    assert ids == part["image_id"].tolist(), "Cache order does not match the partition; rebuild it."
    return np.load(cache_path, mmap_mode="r")


# --------------------------------------------------------------------------- model
def build_model(name, pretrained):
    import torchvision
    w = "IMAGENET1K_V1" if pretrained else None
    if name == "resnet18":
        m = torchvision.models.resnet18(weights=w)
        m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
    elif name == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights=w)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
    else:
        raise ValueError(name)
    return m


class Normalizer:
    def __init__(self, device):
        self.mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)

    def __call__(self, x_uint8):
        x = x_uint8.float().div_(255)
        return ((x - self.mean) / self.std).contiguous(memory_format=torch.channels_last)


def gpu_augment(x_uint8, gen):
    """Dihedral group (flip + 90-degree rotations) and mild brightness/contrast.

    Dermoscopic lesions have no canonical orientation, so all 8 orientations are valid.
    Runs on the GPU so the data pipeline never waits on the 4 Kaggle CPUs.
    """
    x = x_uint8.float().div_(255)
    b = x.shape[0]
    dev = x.device
    flip = torch.rand(b, device=dev, generator=gen) < 0.5
    x = torch.where(flip.view(b, 1, 1, 1), x.flip(3), x)
    k = torch.randint(0, 4, (b,), device=dev, generator=gen)
    out = x.clone()
    for r in (1, 2, 3):
        m = k == r
        if m.any():
            out[m] = torch.rot90(x[m], r, dims=(2, 3))
    bright = (torch.rand(b, 1, 1, 1, device=dev, generator=gen) - 0.5) * 0.2
    contrast = 1 + (torch.rand(b, 1, 1, 1, device=dev, generator=gen) - 0.5) * 0.2
    mean = out.mean(dim=(1, 2, 3), keepdim=True)
    out = ((out - mean) * contrast + mean + bright).clamp_(0, 1)
    return out.mul_(255)  # back to [0, 255] float; Normalizer divides again


# --------------------------------------------------------------------------- metrics
def compute_metrics(y, prob):
    pred = prob.argmax(1)
    k = len(CLASSES)
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    support = cm.sum(1)
    predicted = cm.sum(0)
    recall = np.divide(tp, support, out=np.zeros(k), where=support > 0)
    precision = np.divide(tp, predicted, out=np.zeros(k), where=predicted > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros(k), where=(precision + recall) > 0)
    return {
        "macro_f1": float(f1.mean()),
        "balanced_accuracy": float(recall[support > 0].mean()),
        "accuracy": float(tp.sum() / cm.sum()),
        "per_class": {c: {"precision": float(precision[i]), "recall": float(recall[i]),
                          "f1": float(f1[i]), "support": int(support[i])}
                      for i, c in enumerate(CLASSES)},
        "confusion_matrix": cm.tolist(),
    }


# --------------------------------------------------------------------------- training
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cosine_lr(base, step, total):
    return base * 0.5 * (1 + math.cos(math.pi * step / total))


def make_optimizer(model, lr, args):
    return torch.optim.SGD(model.parameters(), lr=lr, momentum=args.momentum,
                           weight_decay=args.weight_decay, nesterov=True)


def train_one_epoch(model, opt, scaler, X, y, idx, ctx, global_params=None, mu=0.0):
    """One pass over `idx` (a 1-D LongTensor of rows on the device)."""
    model.train()
    perm = idx[torch.randperm(len(idx), device=idx.device, generator=ctx["gen"])]
    bs = ctx["batch_size"]
    total, n = 0.0, 0
    for s in range(0, len(perm), bs):
        bidx = perm[s:s + bs]
        if len(bidx) < 2:  # BatchNorm needs >1 sample
            continue
        xb = ctx["norm"](gpu_augment(X[bidx], ctx["gen"]))
        yb = y[bidx]
        with torch.autocast(device_type=ctx["device_type"], dtype=ctx["amp_dtype"],
                            enabled=ctx["amp"]):
            logits = model(xb)
        loss = F.cross_entropy(logits.float(), yb, weight=ctx["class_weights"])
        if mu > 0:
            prox = sum(((p - g) ** 2).sum() for p, g in zip(model.parameters(), global_params))
            loss = loss + 0.5 * mu * prox
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        total += loss.item() * len(bidx)
        n += len(bidx)
    return total / max(n, 1)


@torch.no_grad()
def predict(model, X, idx, ctx, bs=256):
    model.eval()
    out = []
    for s in range(0, len(idx), bs):
        xb = ctx["norm"](X[idx[s:s + bs]])
        with torch.autocast(device_type=ctx["device_type"], dtype=ctx["amp_dtype"],
                            enabled=ctx["amp"]):
            out.append(model(xb).float().softmax(1))
    return torch.cat(out).cpu().numpy()


def fedavg_aggregate(states, weights):
    agg = {}
    for key, ref in states[0].items():
        if ref.is_floating_point():
            agg[key] = sum(w * s[key].float() for s, w in zip(states, weights)).to(ref.dtype)
        else:  # e.g. BatchNorm num_batches_tracked
            agg[key] = ref.clone()
    return agg


def clone_state(model):
    return {k: v.detach().clone() for k, v in model.state_dict().items()}


def run_name_for(args):
    base = f"{args.arm}_{args.backbone}_s{args.seed}"
    if args.arm == "local":
        base = f"local-{args.client}_{args.backbone}_s{args.seed}"
    if args.arm == "fedprox":
        base = f"fedprox-mu{args.mu:g}_{args.backbone}_s{args.seed}"
    return args.run_name or base


def env_info(device):
    info = {"python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}
    try:
        import torchvision
        info["torchvision"] = torchvision.__version__
    except ImportError:
        pass
    if device.type == "cuda":
        info["gpu"] = torch.cuda.get_device_name(device)
        info["cuda"] = torch.version.cuda
    return info


def main(args):
    if args.build_cache:
        build_cache(args.partition, args.image_root, args.cache)
        return

    run_dir = Path(args.out_dir) / run_name_for(args)
    if (run_dir / "metrics.json").exists():
        print(f"[skip] {run_dir.name} already finished")
        return
    run_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    set_seed(args.seed)

    part = load_partition(args.partition)
    X_np = load_cache(args.cache, part)
    if args.debug_n:  # smoke test: a few images per client x split
        part = part.groupby(["client", "split"], group_keys=False).head(args.debug_n)
    rows = part.index.to_numpy()
    X = torch.from_numpy(np.ascontiguousarray(X_np[rows])).to(device)  # uint8 on the GPU (~1.5 GB)
    part = part.reset_index(drop=True)
    y = torch.tensor(part["dx"].map(CLASSES.index).to_numpy(), device=device)

    def rows_of(mask):
        return torch.tensor(np.flatnonzero(mask.to_numpy()), device=device)

    train_mask = part["split"] == "train"
    val_idx, test_idx = rows_of(part["split"] == "val"), rows_of(part["split"] == "test")
    client_idx = {c: rows_of(train_mask & (part["client"] == c)) for c in CLIENTS}
    pooled_idx = rows_of(train_mask)

    # Class weights: inverse square-root frequency of the pooled training split,
    # normalized to mean 1, identical for every arm.
    counts = np.bincount(part.loc[train_mask, "dx"].map(CLASSES.index), minlength=len(CLASSES))
    cw = 1.0 / np.sqrt(np.maximum(counts, 1))
    cw = cw / cw.mean()

    gen = torch.Generator(device=device)
    gen.manual_seed(args.seed)
    amp = device.type == "cuda"
    ctx = {"gen": gen, "batch_size": args.batch_size, "norm": Normalizer(device),
           "device_type": device.type, "amp": amp, "amp_dtype": torch.float16,
           "class_weights": torch.tensor(cw, dtype=torch.float32, device=device)}
    try:
        scaler = torch.amp.GradScaler(device.type, enabled=amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=amp)

    model = build_model(args.backbone, pretrained=not args.no_pretrained).to(device)
    model = model.to(memory_format=torch.channels_last)

    federated = args.arm in ("fedavg", "fedprox", "fedbn")
    fedbn = args.arm == "fedbn"
    bn_keys, client_bn = set(), {}
    if federated:
        assert args.epochs % args.local_epochs == 0, "epochs must be a multiple of local_epochs"
        n_steps = args.epochs // args.local_epochs  # rounds R
        mu = args.mu if args.arm == "fedprox" else 0.0
        sizes = np.array([len(client_idx[c]) for c in CLIENTS], dtype=float)
        agg_w = (sizes / sizes.sum()).tolist()
        if fedbn:
            # FedBN (Li et al., ICLR 2021): BatchNorm weights, biases and running statistics stay
            # on each client and are never averaged; every other parameter is averaged as in FedAvg.
            for name, mod in model.named_modules():
                if isinstance(mod, nn.modules.batchnorm._BatchNorm):
                    bn_keys.update(f"{name}.{k}" for k in mod.state_dict())
            init = model.state_dict()
            client_bn = {c: {k: init[k].detach().clone() for k in bn_keys} for c in CLIENTS}
    else:
        n_steps = args.epochs
        train_idx = client_idx[args.client] if args.arm == "local" else pooled_idx
        opt = make_optimizer(model, args.lr, args)

    def predict_clients(idx):
        """FedBN: one model per client = shared weights + that client's BatchNorm."""
        shared = clone_state(model)
        out = []
        for c in CLIENTS:
            model.load_state_dict(client_bn[c], strict=False)
            out.append(predict(model, X, idx, ctx))
        model.load_state_dict(shared)
        return np.stack(out)  # (n_clients, n_images, n_classes)

    # ---- resume -------------------------------------------------------------
    start, best_f1, best_step, history = 0, -1.0, -1, []
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
    if last_path.exists():
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        if not federated:
            opt.load_state_dict(ck["opt"])
        if fedbn:
            client_bn = {c: {k: v.to(device) for k, v in bn.items()} for c, bn in ck["client_bn"].items()}
        scaler.load_state_dict(ck["scaler"])
        gen.set_state(ck["gen"])
        torch.set_rng_state(ck["torch_rng"])
        start, best_f1, best_step, history = ck["step"], ck["best_f1"], ck["best_step"], ck["history"]
        print(f"[resume] {run_dir.name} from step {start}")

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    y_val = y[val_idx].cpu().numpy()

    for step in range(start, n_steps):
        lr = cosine_lr(args.lr, step, n_steps)
        ts = time.time()
        if federated:
            global_state = clone_state(model)
            global_params = [p.detach().clone() for p in model.parameters()] if mu > 0 else None
            states, losses = [], []
            for c in CLIENTS:
                model.load_state_dict(global_state)
                if fedbn:
                    model.load_state_dict(client_bn[c], strict=False)
                copt = make_optimizer(model, lr, args)  # fresh client optimizer each round
                for _ in range(args.local_epochs):
                    losses.append(train_one_epoch(model, copt, scaler, X, y, client_idx[c], ctx,
                                                  global_params, mu))
                st = clone_state(model)
                states.append(st)
                if fedbn:
                    client_bn[c] = {k: st[k] for k in bn_keys}
            # FedBN: BN entries of the averaged state are never used (each client keeps its own).
            model.load_state_dict(fedavg_aggregate(states, agg_w))
            del states
            train_loss = float(np.mean(losses))
        else:
            for g in opt.param_groups:
                g["lr"] = lr
            train_loss = train_one_epoch(model, opt, scaler, X, y, train_idx, ctx)

        if fedbn:  # selection score = mean pooled-val macro-F1 of the 4 client models
            per_client = [compute_metrics(y_val, p) for p in predict_clients(val_idx)]
            vm = {k: float(np.mean([m[k] for m in per_client]))
                  for k in ("macro_f1", "balanced_accuracy", "accuracy")}
        else:
            vm = compute_metrics(y_val, predict(model, X, val_idx, ctx))
        improved = vm["macro_f1"] > best_f1
        if improved:
            best_f1, best_step = vm["macro_f1"], step + 1
            torch.save({"model": model.state_dict(), "client_bn": client_bn} if fedbn
                       else model.state_dict(), best_path)
        rec = {"step": step + 1, "lr": lr, "train_loss": train_loss,
               "val_macro_f1": vm["macro_f1"], "val_bal_acc": vm["balanced_accuracy"],
               "val_acc": vm["accuracy"], "seconds": time.time() - ts}
        if fedbn:
            rec.update({f"val_macro_f1_{c}": m["macro_f1"] for c, m in zip(CLIENTS, per_client)})
        history.append(rec)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        print(f"{run_dir.name} {'round' if federated else 'epoch'} {step + 1}/{n_steps} "
              f"loss {train_loss:.4f} val-MF1 {vm['macro_f1']:.4f}{' *' if improved else ''} "
              f"({time.time() - ts:.0f}s)", flush=True)
        torch.save({"model": model.state_dict(), "opt": None if federated else opt.state_dict(),
                    "client_bn": client_bn if fedbn else None,
                    "scaler": scaler.state_dict(), "gen": gen.get_state(),
                    "torch_rng": torch.get_rng_state(), "step": step + 1, "best_f1": best_f1,
                    "best_step": best_step, "history": history}, last_path)

    # ---- final evaluation with the best checkpoint (test used once) ----------
    best = torch.load(best_path, map_location=device, weights_only=False)
    y_test = y[test_idx].cpu().numpy()
    if fedbn:
        model.load_state_dict(best["model"])
        client_bn = best["client_bn"]
        val_prob, test_prob = predict_clients(val_idx), predict_clients(test_idx)
        per_model = {c: {"val": compute_metrics(y_val, val_prob[i]), "test": compute_metrics(y_test, test_prob[i])}
                     for i, c in enumerate(CLIENTS)}
        summary = lambda split: {k: float(np.mean([per_model[c][split][k] for c in CLIENTS]))
                                 for k in ("macro_f1", "balanced_accuracy", "accuracy")}
        val_m, test_m = summary("val"), summary("test")
    else:
        model.load_state_dict(best)
        val_prob, test_prob = predict(model, X, val_idx, ctx), predict(model, X, test_idx, ctx)
        val_m, test_m, per_model = compute_metrics(y_val, val_prob), compute_metrics(y_test, test_prob), None
    np.savez_compressed(
        run_dir / "preds.npz",
        val_ids=part.loc[val_idx.cpu().numpy(), "image_id"].to_numpy(), val_y=y_val, val_prob=val_prob,
        test_ids=part.loc[test_idx.cpu().numpy(), "image_id"].to_numpy(), test_y=y_test,
        test_prob=test_prob, clients=np.array(CLIENTS if fedbn else []))

    metrics = {
        "run": run_dir.name, "arm": args.arm, "client": args.client if args.arm == "local" else None,
        "backbone": args.backbone, "seed": args.seed,
        "mu": args.mu if args.arm == "fedprox" else (0.0 if federated else None),
        "epochs_budget": args.epochs,
        "rounds": n_steps if federated else None,
        "local_epochs": args.local_epochs if federated else None,
        "lr": args.lr, "momentum": args.momentum, "weight_decay": args.weight_decay,
        "batch_size": args.batch_size, "optimizer": "SGD nesterov, cosine schedule",
        "loss": "cross-entropy, inverse-sqrt class weights (pooled train)",
        "class_weights": dict(zip(CLASSES, map(float, cw))),
        "n_train_images": int(sum(len(v) for v in client_idx.values())) if federated or args.arm == "centralized"
        else int(len(client_idx[args.client])),
        "best_step": best_step,
        "selection": "best mean pooled-val macro-F1 over client models" if fedbn else "best pooled-val macro-F1",
        "val": val_m, "test": test_m,
        "per_client_model": per_model,
        "train_seconds": time.time() - t0,
        "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated(device) / 1e9) if device.type == "cuda" else None,
        "debug_n": args.debug_n, "env": env_info(device),
    }
    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    last_path.unlink(missing_ok=True)
    print(f"[done] {run_dir.name}: best {('round' if federated else 'epoch')} {best_step}, "
          f"val MF1 {metrics['val']['macro_f1']:.4f}, test MF1 {metrics['test']['macro_f1']:.4f}, "
          f"{metrics['train_seconds'] / 60:.1f} min")


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--build-cache", action="store_true")
    p.add_argument("--image-root", default="/kaggle/input")
    p.add_argument("--partition", required=True)
    p.add_argument("--cache", required=True)
    p.add_argument("--out-dir", default="runs")
    p.add_argument("--run-name", default=None)
    p.add_argument("--arm", choices=["centralized", "local", "fedavg", "fedprox", "fedbn"], default="centralized")
    p.add_argument("--client", choices=CLIENTS, default=None)
    p.add_argument("--backbone", choices=["resnet18", "efficientnet_b0"], default="resnet18")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--epochs", type=int, default=20, help="budget: centralized epochs = R x E")
    p.add_argument("--local-epochs", type=int, default=1, help="E (federated arms)")
    p.add_argument("--mu", type=float, default=0.0, help="FedProx proximal weight")
    p.add_argument("--lr", type=float, default=0.01)
    p.add_argument("--momentum", type=float, default=0.9)
    p.add_argument("--weight-decay", type=float, default=1e-4)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--no-pretrained", action="store_true")
    p.add_argument("--debug-n", type=int, default=0, help="smoke test: images per client x split")
    a = p.parse_args(argv)
    if a.arm == "local" and a.client is None:
        p.error("--arm local needs --client")
    if a.arm == "fedbn":
        a.mu = 0.0
    if a.arm == "fedprox" and a.mu <= 0:
        p.error("--arm fedprox needs --mu > 0")
    return a


if __name__ == "__main__":
    main(parse_args())

Writing fl_train.py


## 2. Verify partition · build cache · download weights

In [3]:
import importlib
import fl_train
importlib.reload(fl_train)
from fl_train import CLASSES, CLIENTS, FL_COLUMNS, FL_MANIFEST_SHA256, fingerprint

part = pd.read_csv(PARTITION_PATH)
assert fingerprint(part, FL_COLUMNS) == FL_MANIFEST_SHA256, "Partition fingerprint mismatch"
assert part["split"].value_counts().to_dict() == {"train": 6981, "val": 1532, "test": 1502}
print("FL partition verified")

if not CACHE_PATH.exists():
    subprocess.run([sys.executable, "fl_train.py", "--build-cache", "--partition", str(PARTITION_PATH),
                    "--image-root", str(INPUT_ROOT), "--cache", str(CACHE_PATH)], check=True)
if not DEBUG:
    import torchvision
    torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1")

FL partition verified
  cached 2,000/10,015 (9s)
  cached 4,000/10,015 (18s)
  cached 6,000/10,015 (26s)
  cached 8,000/10,015 (35s)
  cached 10,000/10,015 (44s)
Cache written: /kaggle/working/cache/ham10000_224_uint8.npy (10015, 3, 224, 224) in 45s
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 171MB/s]


## 3. Runner + smoke test

In [4]:
COMMON = ["--partition", str(PARTITION_PATH), "--cache", str(CACHE_PATH), "--out-dir", str(RUNS_DIR),
          "--epochs", str(EPOCHS), "--local-epochs", str(LOCAL_EPOCHS), *EXTRA_ARGS]

def run_name(job):
    return fl_train.run_name_for(fl_train.parse_args(COMMON + job))

def run_jobs(jobs, stage):
    q = queue.Queue()
    for j in jobs:
        q.put(j)
    failures, lock, t_stage = [], threading.Lock(), time.time()

    def worker(slot):
        while True:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            name = run_name(job)
            env = dict(os.environ)
            if N_GPUS:
                env["CUDA_VISIBLE_DEVICES"] = str(slot)
            t0 = time.time()
            with open(LOG_DIR / f"{name}.log", "a") as log:
                rc = subprocess.run([sys.executable, "fl_train.py", *COMMON, *job],
                                    stdout=log, stderr=subprocess.STDOUT, env=env).returncode
            tail = (LOG_DIR / f"{name}.log").read_text().strip().splitlines()[-1:]
            with lock:
                print(f"[{stage}][gpu{slot}] {'OK  ' if rc == 0 else 'FAIL'} {name:34s} "
                      f"{(time.time() - t0) / 60:5.1f} min | {tail[0] if tail else ''}", flush=True)
                if rc:
                    failures.append(name)

    threads = [threading.Thread(target=worker, args=(s,)) for s in range(SLOTS)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    print(f"[{stage}] finished in {(time.time() - t_stage) / 60:.1f} min")
    if failures:
        raise RuntimeError(f"Failed runs: {failures}. See {LOG_DIR}/<run>.log")

smoke = WORK / "smoke"
shutil.rmtree(smoke, ignore_errors=True)
args = [*COMMON, "--arm", "fedbn", "--backbone", BACKBONE, "--out-dir", str(smoke), "--epochs", "2", "--debug-n", "16"]
r = subprocess.run([sys.executable, "fl_train.py", *args, *(["--no-pretrained"] if DEBUG else [])],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-2000:])
assert r.returncode == 0, "Smoke test failed"
shutil.rmtree(smoke, ignore_errors=True)
print("Smoke test passed")

fedbn_efficientnet_b0_s42 round 1/2 loss 1.9613 val-MF1 0.1203 * (35s)
fedbn_efficientnet_b0_s42 round 2/2 loss 1.8581 val-MF1 0.0996 (1s)
[done] fedbn_efficientnet_b0_s42: best round 1, val MF1 0.1203, test MF1 0.1155, 0.6 min
 
Smoke test passed


## 4. Train FedBN (3 seeds, two GPUs in parallel)

In [5]:
run_jobs([["--arm", "fedbn", "--backbone", BACKBONE, "--seed", str(s)] for s in SEEDS], "FedBN")

[FedBN][gpu1] OK   fedbn_efficientnet_b0_s43            9.1 min | [done] fedbn_efficientnet_b0_s43: best round 17, val MF1 0.5785, test MF1 0.5598, 9.0 min
[FedBN][gpu0] OK   fedbn_efficientnet_b0_s42           10.0 min | [done] fedbn_efficientnet_b0_s42: best round 17, val MF1 0.5869, test MF1 0.5399, 9.9 min
[FedBN][gpu1] OK   fedbn_efficientnet_b0_s44            9.3 min | [done] fedbn_efficientnet_b0_s44: best round 19, val MF1 0.5651, test MF1 0.5431, 9.2 min
[FedBN] finished in 18.4 min


## 5. Results

Three views, all on the same frozen test split:

- **A. Pooled test** (same yardstick as the main table): each FedBN client model is scored on all 1,502 test images, then averaged over clients - exactly like Local-only.
- **B. Each client's own test images** (the FedBN paper's evaluation): client *k*'s model on client *k*'s test images. FedAvg / Centralized use their single model on the same images. Macro-F1 here averages only the classes present in that client's test set, and several cells are tiny (Sec. III of the proposal), so read B as indicative.
- **C. Per-class recall** on the pooled test, FedBN (mean of client models) vs FedAvg.

In [6]:
def load_preds(run_dir):
    z = np.load(run_dir / "preds.npz", allow_pickle=True)
    return z["test_ids"], z["test_y"], z["test_prob"]

fb_ids, fb_y, _ = load_preds(RUNS_DIR / f"fedbn_{BACKBONE}_s{SEEDS[0]}")
test_part = part.set_index("image_id").loc[fb_ids].reset_index()
K = len(CLASSES)

def metr(y, pred, present_only=False):
    cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K)
    tp, sup, prd = np.diag(cm).astype(float), cm.sum(1), cm.sum(0)
    rec = np.divide(tp, sup, out=np.zeros(K), where=sup > 0)
    prec = np.divide(tp, prd, out=np.zeros(K), where=prd > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    keep = sup > 0 if present_only else np.ones(K, bool)
    return {"macro_f1": f1[keep].mean(), "bal_acc": rec[sup > 0].mean(), "acc": tp.sum() / cm.sum(),
            **{f"recall_{c}": rec[i] for i, c in enumerate(CLASSES)}}

# FedBN predictions: arm -> seed -> {client: predicted labels on fb_ids}
fedbn = {}
for s in SEEDS:
    ids, y, prob = load_preds(RUNS_DIR / f"fedbn_{BACKBONE}_s{s}")
    assert (ids == fb_ids).all()
    fedbn[s] = {c: prob[i].argmax(1) for i, c in enumerate(CLIENTS)}

prev = {}  # name -> seed -> {client or "all": predicted labels aligned to fb_ids}
if PREV_RUNS:
    def aligned(run):
        ids, _, prob = load_preds(PREV_RUNS / run)
        pos = pd.Series(np.arange(len(ids)), index=ids).loc[fb_ids].to_numpy()
        return prob[pos].argmax(1)
    for s in SEEDS:
        try:
            prev.setdefault("Centralized", {})[s] = {"all": aligned(f"centralized_{BACKBONE}_s{s}")}
            prev.setdefault("FedAvg", {})[s] = {"all": aligned(f"fedavg_{BACKBONE}_s{s}")}
            prev.setdefault("Local-only", {})[s] = {c: aligned(f"local-{c}_{BACKBONE}_s{s}") for c in CLIENTS}
        except FileNotFoundError as e:
            print(f"Missing notebook-03 run for seed {s}: {e}")
ARMS = {"FedBN (client models)": fedbn, **prev}

def model_for(arm_preds, seed, client):
    d = arm_preds[seed]
    return d["all"] if "all" in d else d[client]

seeds_ok = [s for s in SEEDS if all(s in a for a in ARMS.values())]

# ---- A. pooled test ------------------------------------------------------------
rowsA = []
for arm, P in ARMS.items():
    per_seed = [np.mean([metr(fb_y, p)["macro_f1"] for p in P[s].values()]) for s in seeds_ok]
    per_seed_bal = [np.mean([metr(fb_y, p)["bal_acc"] for p in P[s].values()]) for s in seeds_ok]
    rowsA.append({"arm": arm, "models_per_seed": len(P[seeds_ok[0]]), "test_macro_f1": np.mean(per_seed),
                  "sd_over_seeds": np.std(per_seed, ddof=1) if len(per_seed) > 1 else np.nan,
                  "test_bal_acc": np.mean(per_seed_bal)})
A = pd.DataFrame(rowsA).set_index("arm")

# lesion bootstrap: FedBN (mean of client models) - FedAvg, paired
if "FedAvg" in prev:
    codes, _ = pd.factorize(test_part["lesion_id"])
    groups = [np.flatnonzero(codes == g) for g in range(codes.max() + 1)]
    rng = np.random.default_rng(2026)
    boots = [np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))]) for _ in range(N_BOOTSTRAP)]
    def boot(P):
        preds = [p for s in seeds_ok for p in P[s].values()]
        return np.array([np.mean([metr(fb_y[ix], p[ix])["macro_f1"] for p in preds]) for ix in boots])
    b_fb, b_fa = boot(fedbn), boot(prev["FedAvg"])
    A.loc["FedBN (client models)", "ci_low"], A.loc["FedBN (client models)", "ci_high"] = np.percentile(b_fb, [2.5, 97.5])
    d = b_fb - b_fa
    A.loc["FedBN (client models)", "gap_vs_FedAvg"] = A.loc["FedBN (client models)", "test_macro_f1"] - A.loc["FedAvg", "test_macro_f1"]
    A.loc["FedBN (client models)", "gap_ci_low"], A.loc["FedBN (client models)", "gap_ci_high"] = np.percentile(d, [2.5, 97.5])
A.to_csv(RESULTS_DIR / "fedbn_A_pooled_test.csv")
print(f"A. Pooled test ({BACKBONE}, seeds {seeds_ok}); FedBN and Local-only = mean over the 4 client models")
display(A.round(4))

# ---- B. each client's own test images ----------------------------------------------
rowsB = []
for c in CLIENTS:
    m = (test_part["client"] == c).to_numpy()
    row = {"client": c, "test_images": int(m.sum()),
           "classes_in_test": int((np.bincount(fb_y[m], minlength=K) > 0).sum())}
    for arm, P in ARMS.items():
        sc = [metr(fb_y[m], model_for(P, s, c)[m], present_only=True) for s in seeds_ok]
        row[f"{arm} | macro_f1"] = np.mean([x["macro_f1"] for x in sc])
        row[f"{arm} | bal_acc"] = np.mean([x["bal_acc"] for x in sc])
    rowsB.append(row)
B = pd.DataFrame(rowsB).set_index("client")
B.to_csv(RESULTS_DIR / "fedbn_B_per_client_test.csv")
print("B. Each client's model on its own test images (macro-F1 over classes present)")
display(B.round(3))

# ---- C. per-class recall, pooled test ---------------------------------------------
C_ = pd.DataFrame({arm: pd.DataFrame([metr(fb_y, p) for s in seeds_ok for p in P[s].values()])
                   [[f"recall_{c}" for c in CLASSES]].mean() for arm, P in ARMS.items()}).T
C_.columns = CLASSES
C_.to_csv(RESULTS_DIR / "fedbn_C_per_class_recall.csv")
print("C. Per-class recall on the pooled test")
display(C_.round(3))

hist = pd.concat([pd.read_csv(RUNS_DIR / f"fedbn_{BACKBONE}_s{s}" / "history.csv").assign(seed=s) for s in SEEDS])
hist.to_csv(RESULTS_DIR / "fedbn_val_history.csv", index=False)
print({s: json.loads((RUNS_DIR / f"fedbn_{BACKBONE}_s{s}" / "metrics.json").read_text())["best_step"] for s in SEEDS},
      "= best round per seed")

A. Pooled test (efficientnet_b0, seeds [42, 43, 44]); FedBN and Local-only = mean over the 4 client models


,models_per_seed,test_macro_f1,sd_over_seeds,test_bal_acc,ci_low,ci_high,gap_vs_FedAvg,gap_ci_low,gap_ci_high
arm,,,,,,,,,
FedBN (client models),4,0.5476,0.0107,0.5698,0.5007,0.582,-0.0764,-0.107,-0.0507
Centralized,1,0.7154,0.0124,0.7059,NaN,NaN,NaN,NaN,NaN
FedAvg,1,0.6240,0.0057,0.6337,NaN,NaN,NaN,NaN,NaN
Local-only,4,0.3798,0.0051,0.4140,NaN,NaN,NaN,NaN,NaN


B. Each client's model on its own test images (macro-F1 over classes present)


,test_images,classes_in_test,FedBN (client models) | macro_f1,FedBN (client models) | bal_acc,Centralized | macro_f1,Centralized | bal_acc,FedAvg | macro_f1,FedAvg | bal_acc,Local-only | macro_f1,Local-only | bal_acc
client,,,,,,,,,,
vidir_molemax,600,6,0.692,0.780,0.757,0.743,0.679,0.758,0.567,0.610
vidir_modern,506,7,0.599,0.614,0.657,0.738,0.549,0.612,0.618,0.595
rosendahl,330,7,0.683,0.738,0.619,0.629,0.611,0.628,0.544,0.581
vienna_dias,66,5,0.449,0.546,0.435,0.515,0.382,0.555,0.233,0.248


C. Per-class recall on the pooled test


,nv,mel,bkl,bcc,akiec,vasc,df
FedBN (client models),0.904,0.345,0.470,0.515,0.397,0.683,0.675
Centralized,0.924,0.593,0.645,0.775,0.705,0.667,0.633
FedAvg,0.916,0.273,0.643,0.685,0.385,0.667,0.867
Local-only,0.821,0.369,0.445,0.359,0.199,0.397,0.308


{42: 17, 43: 17, 44: 19} = best round per seed


## 6. LaTeX table (per-client view) + download bundle

In [7]:
arms = list(ARMS)
short = {"FedBN (client models)": "FedBN", "Centralized": "Central.", "FedAvg": "FedAvg", "Local-only": "Local"}
head = "Client & $n$ & " + " & ".join(short[a] for a in arms) + " \\\\"
body = [f"{c.replace('_', chr(92) + '_')} & {int(B.loc[c, 'test_images'])} & "
        + " & ".join(f"{B.loc[c, a + ' | macro_f1']:.3f}" for a in arms) + " \\\\" for c in CLIENTS]
latex = "\n".join(["\\begin{table}[t]",
                   "\\caption{Secondary analysis: Macro-F1 of each client's model on that client's own test images "
                   f"(classes present only; mean of {len(seeds_ok)} seeds). FedBN keeps BatchNorm local.}}",
                   "\\label{tab:fedbn}", "\\centering", "\\footnotesize",
                   "\\begin{tabular}{lr" + "c" * len(arms) + "}", "\\hline", head, "\\hline", *body,
                   "\\hline", "\\end{tabular}", "\\end{table}"])
(RESULTS_DIR / "table_fedbn_per_client.tex").write_text(latex + "\n", encoding="utf-8")
print(latex)

bundle = WORK / "bundle_fedbn"
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(RESULTS_DIR, bundle / "results_fedbn")
shutil.copytree(LOG_DIR, bundle / "logs")
shutil.copy("fl_train.py", bundle / "fl_train.py")
for s in SEEDS:
    d = RUNS_DIR / f"fedbn_{BACKBONE}_s{s}"
    (bundle / "runs" / d.name).mkdir(parents=True)
    for f in ("metrics.json", "history.csv", "preds.npz"):
        shutil.copy(d / f, bundle / "runs" / d.name / f)
archive = shutil.make_archive(str(WORK / "ham10000_fedbn_results"), "zip", bundle)
shutil.rmtree(bundle)
print(f"Download: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")

\begin{table}[t]
\caption{Secondary analysis: Macro-F1 of each client's model on that client's own test images (classes present only; mean of 3 seeds). FedBN keeps BatchNorm local.}
\label{tab:fedbn}
\centering
\footnotesize
\begin{tabular}{lrcccc}
\hline
Client & $n$ & FedBN & Central. & FedAvg & Local \\
\hline
vidir\_molemax & 600 & 0.692 & 0.757 & 0.679 & 0.567 \\
vidir\_modern & 506 & 0.599 & 0.657 & 0.549 & 0.618 \\
rosendahl & 330 & 0.683 & 0.619 & 0.611 & 0.544 \\
vienna\_dias & 66 & 0.449 & 0.435 & 0.382 & 0.233 \\
\hline
\end{tabular}
\end{table}
Download: /kaggle/working/ham10000_fedbn_results.zip (1.0 MB)


## Notes for the report

- FedBN is a **secondary analysis**, not a fifth arm of the main table: it yields client-specific models, so its pooled-test score is the mean over 4 client models (like Local-only), and its natural evaluation is view B.
- Model selection used the pooled validation set (mean over client models), the same simulation assumption as the other arms.
- BatchNorm running statistics are averaged in plain FedAvg; in FedBN they are local. If FedBN helps mostly in view B, the gain comes from adapting to each source's image statistics (feature shift), not from better global classification.